In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from mg_sampling.density.grid import Grid
import MDAnalysis as mda
from MDAnalysis.lib.transformations import rotation_matrix
import numpy as np
import matplotlib.pyplot as plt

from mg_sampling import PROJECT_DIR
from mg_sampling.density.utils import indices_to_mask

## Align two maps

In [ ]:
old_map = Grid(PROJECT_DIR / 'density/emd_31385.map', file_format='mrc')
new_map = Grid(PROJECT_DIR / 'density/emd_42499.map', file_format='mrc')
old_u = mda.Universe(PROJECT_DIR / 'density/7ez0.pdb')
new_u = mda.Universe(PROJECT_DIR / 'density/9cbu.pdb')

In [ ]:
# from scipy.spatial.transform import Rotation

# R0 = Rotation.random().as_matrix()

# Test to see if it works with a random initial rotation
# old_map = old_map.rotate(R0, point=old_u.atoms.center_of_mass())
# old_u.atoms.rotate(R0, point=old_u.atoms.center_of_mass())

In [ ]:
from mg_sampling.density.align import reorder_residue_atoms
new_u.select_atoms('name O1P').names = 'OP1'
new_u.select_atoms('name O2P').names = 'OP2'
old_ag = reorder_residue_atoms(old_u.select_atoms('nucleic and not type H'))
new_ag = reorder_residue_atoms(new_u.select_atoms('nucleic and not type H and not (resid 22 and name P OP1 OP2)'))
assert np.all(old_ag.names == new_ag.names)

In [ ]:
from mg_sampling.density.utils import voxels_around
map_params = {'dx': new_map.delta, 'x0': new_map.origin, 'nbin': new_map.shape}
vox_indices = voxels_around(new_ag, 8, map_params)
new_cc_mask = indices_to_mask(vox_indices, map_params)

map_params = {'dx': old_map.delta, 'x0': old_map.origin, 'nbin': old_map.shape}
vox_indices = voxels_around(old_ag, 8, map_params)
old_cc_mask = indices_to_mask(vox_indices, map_params)

In [ ]:
from matplotlib.colors import CenteredNorm

fig, (ax1, ax2) = plt.subplots(ncols=2, sharex=True, sharey=True, figsize=(12, 4))
cmap = 'RdBu_r'

ag = new_ag
m = new_map
cc_mask = new_cc_mask
ax = ax1
norm = CenteredNorm()
midpoints = m.midpoints[0]
L = midpoints[-1] - midpoints[0]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
mask = (ag.positions[:, 2] > L / 2 - 2) & (ag.positions[:, 2] < L / 2 + 2)
im = ax.imshow(m.grid[:, :, len(m.grid)//2].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
plt.colorbar(im, ax=ax)
ax.contour(cc_mask[:, :, len(m.grid)//2].T, origin='lower', extent=extent)
ax.set_title('New map')

ag = old_ag
m = old_map
cc_mask = old_cc_mask
ax = ax2
norm = CenteredNorm()
midpoints = m.midpoints[0]
L = midpoints[-1] - midpoints[0]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
mask = (ag.positions[:, 2] > L / 2 - 2) & (ag.positions[:, 2] < L / 2 + 2)
im = ax.imshow(m.grid[:, :, len(m.grid)//2].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
#ax.scatter(new_ag.positions[mask, 0], new_ag.positions[mask, 1], s=1, color='gray', marker='.')
plt.colorbar(im, ax=ax)
ax.contour(cc_mask[:, :, len(m.grid)//2].T, origin='lower', extent=extent)
ax.set_title('Old map')

for ax in (ax1, ax2):
    ax.set_xlabel('x [Å]')
    ax.set_ylabel('y [Å]')


In [ ]:
# Align old to new with structures
vector = new_ag.center_of_mass() - old_ag.center_of_mass()
R, _ = mda.analysis.align.rotation_matrix(old_ag.positions - old_ag.center_of_mass(), new_ag.positions - new_ag.center_of_mass())
point = new_ag.center_of_mass()

old_ag_aligned = mda.Merge(old_ag).atoms.translate(vector).rotate(R, point=point)

new_ag_aligned = mda.Merge(new_ag).atoms.rotate(R.T, point=point).translate(-vector)

In [ ]:
# Same for map with resample
old_map_aligned = old_map.translate_origin(vector).resample(new_map).rotate(R, point=point)
# Also align new to old
new_map_aligned = new_map.rotate(R.T, point=point).translate_origin(-vector).resample(old_map)

In [ ]:
# Recompute cc_mask for aligned maps
map_params = {'dx': old_map_aligned.delta, 'x0': old_map_aligned.origin, 'nbin': old_map_aligned.shape}
vox_indices = voxels_around(old_ag_aligned, 10, map_params)
old_cc_mask_aligned = indices_to_mask(vox_indices, map_params)

map_params = {'dx': new_map_aligned.delta, 'x0': new_map_aligned.origin, 'nbin': new_map_aligned.shape}
vox_indices = voxels_around(new_ag_aligned, 10, map_params)
new_cc_mask_aligned = indices_to_mask(vox_indices, map_params)

In [ ]:
from matplotlib.colors import CenteredNorm

fig, axarr = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True, figsize=(12, 12))
cmap = 'RdBu_r'

ag = new_ag
m = new_map
cc_mask = new_cc_mask
ax = axarr[0, 0]
k = len(m.midpoints[2])//2
z = m.midpoints[2][k]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
norm = CenteredNorm()
mask = (ag.positions[:, 2] > z - 2) & (ag.positions[:, 2] < z + 2)
im = ax.imshow(m.grid[:, :, k].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
ax.contour(cc_mask[:, :, k].T, origin='lower', extent=extent)
plt.colorbar(im, ax=ax)

ag = old_ag
m = old_map
cc_mask = old_cc_mask
ax = axarr[0, 1]
k = len(m.midpoints[2])//2
z = m.midpoints[2][k]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
norm = CenteredNorm()
mask = (ag.positions[:, 2] > z - 2) & (ag.positions[:, 2] < z + 2)
im = ax.imshow(m.grid[:, :, k].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
ax.contour(cc_mask[:, :, k].T, origin='lower', extent=extent)
plt.colorbar(im, ax=ax)

ag = new_ag
m = new_map
cc_mask = new_cc_mask
ax = axarr[1, 0]
k = len(m.midpoints[2])//2
z = m.midpoints[2][k]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
norm = CenteredNorm()
mask = (ag.positions[:, 2] > z - 2) & (ag.positions[:, 2] < z + 2)
im = ax.imshow(m.grid[:, :, k].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
ax.contour(cc_mask[:, :, k].T, origin='lower', extent=extent)
plt.colorbar(im, ax=ax)

ag = old_ag_aligned
m = old_map_aligned
cc_mask = old_cc_mask_aligned
ax = axarr[1, 1]
k = len(m.midpoints[2])//2
z = m.midpoints[2][k]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
norm = CenteredNorm()
mask = (ag.positions[:, 2] > z - 2) & (ag.positions[:, 2] < z + 2)
im = ax.imshow(m.grid[:, :, k].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
ax.contour(cc_mask[:, :, k].T, origin='lower', extent=extent)
plt.colorbar(im, ax=ax)

plt.xlim(50, 150)
plt.ylim(50, 150)
for ax in (ax1, ax2):
    ax.set_xlabel('x [Å]')
    ax.set_ylabel('y [Å]')


In [ ]:
## Unsharpened maps

tempfactors = {
    'old': -14.0,
    'new': -79.23,
}

new_map_unsharp = new_map.apply_tempfactor(-tempfactors['new'])
old_map_unsharp = old_map.apply_tempfactor(-tempfactors['old'])
new_map_aligend_unsharp = new_map_aligned.apply_tempfactor(-tempfactors['new'])
old_map_aligned_unsharp = old_map_aligned.apply_tempfactor(-tempfactors['old'])


In [ ]:
from matplotlib.colors import CenteredNorm

fig, axarr = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True, figsize=(12, 12))
cmap = 'RdBu_r'

ag = new_ag
m = new_map
cc_mask = new_cc_mask
ax = axarr[0, 0]
k = len(m.midpoints[2])//2
z = m.midpoints[2][k]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
norm = CenteredNorm()
mask = (ag.positions[:, 2] > z - 2) & (ag.positions[:, 2] < z + 2)
im = ax.imshow(m.grid[:, :, k].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
ax.contour(cc_mask[:, :, k].T, origin='lower', extent=extent)
plt.colorbar(im, ax=ax)

ag = old_ag
m = old_map_aligned
cc_mask = old_cc_mask_aligned
ax = axarr[0, 1]
k = len(m.midpoints[2])//2
z = m.midpoints[2][k]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
norm = CenteredNorm()
mask = (ag.positions[:, 2] > z - 2) & (ag.positions[:, 2] < z + 2)
im = ax.imshow(m.grid[:, :, k].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
ax.contour(cc_mask[:, :, k].T, origin='lower', extent=extent)
plt.colorbar(im, ax=ax)

ag = new_ag
m = new_map_unsharp
cc_mask = new_cc_mask
ax = axarr[1, 0]
k = len(m.midpoints[2])//2
z = m.midpoints[2][k]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
norm = CenteredNorm()
mask = (ag.positions[:, 2] > z - 2) & (ag.positions[:, 2] < z + 2)
im = ax.imshow(m.grid[:, :, k].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
ax.contour(cc_mask[:, :, k].T, origin='lower', extent=extent)
plt.colorbar(im, ax=ax)

ag = old_ag
m = old_map_aligned_unsharp
cc_mask = old_cc_mask_aligned
ax = axarr[1, 1]
k = len(m.midpoints[2])//2
z = m.midpoints[2][k]
x_edges, y_edges, z_edges = m.edges
extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
norm = CenteredNorm()
mask = (ag.positions[:, 2] > z - 2) & (ag.positions[:, 2] < z + 2)
im = ax.imshow(m.grid[:, :, k].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')
ax.scatter(ag.positions[mask, 0], ag.positions[mask, 1], s=1, color='black', marker='.')
ax.contour(cc_mask[:, :, k].T, origin='lower', extent=extent)
plt.colorbar(im, ax=ax)

plt.xlim(50, 150)
plt.ylim(50, 150)
for ax in (ax1, ax2):
    ax.set_xlabel('x [Å]')
    ax.set_ylabel('y [Å]')


In [ ]:
from mutual_info import compute_mi

In [ ]:
from scipy.stats import linregress

cc_mask = new_cc_mask & old_cc_mask_aligned

res = linregress(old_map_aligned.grid[cc_mask], new_map.grid[cc_mask])
print("Sharpened maps CC:", res)
print("Sharpened maps MI:", compute_mi(old_map_aligned.grid[cc_mask], new_map.grid[cc_mask]))


res = linregress(
    old_map_aligned_unsharp.grid[cc_mask],
    new_map_unsharp.grid[cc_mask],
)
print("Unsharpened maps CC:", res)
print("Unsharpened maps MI:", compute_mi(old_map_aligned_unsharp.grid[cc_mask], new_map_unsharp.grid[cc_mask]))


### Fix one map sharpening, vary the other

In [ ]:
from tqdm import tqdm
tempfactors = np.linspace(0, 200, 21)

ccs = []
ccs_unsharp = []
ccs_aligned = []
ccs_aligned_unsharp = []
for B in tqdm(tempfactors):
    _map = new_map.apply_tempfactor(B)
    ccs.append(linregress(_map.grid[new_cc_mask], old_map_aligned.grid[new_cc_mask]).rvalue)
    ccs_unsharp.append(linregress(_map.grid[new_cc_mask], old_map_aligned_unsharp.grid[new_cc_mask]).rvalue)
    _map = new_map_aligned.apply_tempfactor(B)
    ccs_aligned.append(linregress(_map.grid[cc_mask], old_map.grid[cc_mask]).rvalue)
    ccs_aligned_unsharp.append(linregress(_map.grid[cc_mask], old_map_unsharp.grid[cc_mask]).rvalue)

In [ ]:
plt.plot(tempfactors, ccs, label="Old map (aligned to new)", color='blue')
plt.plot(tempfactors, ccs_unsharp, label="Old map unsharp. (aligned to new)", color='red')
plt.plot(tempfactors, ccs_aligned, label="Old map (aligned to old)", color='blue', ls='dotted')
plt.plot(tempfactors, ccs_aligned_unsharp, label="Old map unsharp. (aligned to old)", color='red', ls='dotted')

plt.axvline(79.23, ls='dashed', color='k', label="Exp. sharpening")

plt.legend()
plt.title("New map (EMD42499)")
plt.xlabel("Smoothing B-factor")
plt.ylabel("CC")


In [ ]:
from tqdm import tqdm
tempfactors = np.linspace(0, 200, 21)

ccs = []
ccs_unsharp = []
ccs_aligned = []
ccs_aligned_unsharp = []
for B in tqdm(tempfactors):
    _map = old_map.apply_tempfactor(B)
    ccs.append(linregress(_map.grid[cc_mask], new_map_aligned.grid[cc_mask]).rvalue)
    ccs_unsharp.append(linregress(_map.grid[cc_mask], new_map_aligned_unsharp.grid[cc_mask]).rvalue)
    _map = old_map_aligned.apply_tempfactor(B)
    ccs_aligned.append(linregress(_map.grid[cc_mask], new_map.grid[cc_mask]).rvalue)
    ccs_aligned_unsharp.append(linregress(_map.grid[cc_mask], new_map_unsharp.grid[cc_mask]).rvalue)